# SQL 子查询（练习）

由于担心之前的AI对话长度受限，我重新在Claude的Project下新建了一个项目，所以教授的代码语法习惯会有些不同。但从这周开始就会保持统一了。

## 0. 环境 + 看清表结构

- `%load_ext sql` 加载 jupysql,notebook 才认识 `%sql` / `%%sql`(重启 kernel 后要重跑)。
- `%sql duckdb:///:memory:` 开一个**内存数据库**,关掉就清空,适合练习。
- `%` 管一行,`%%` 管一格;`%%sql` 表示整格内容是 SQL。

In [7]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


- `DESCRIBE` 看数据结构(≈ Pandas `df.info()`),碰新数据第一件事;
  jupysql 渲染不出 `DESCRIBE` 时,用 `SELECT * FROM (DESCRIBE ...)` 包一层。

In [8]:
%%sql
SELECT * FROM (DESCRIBE SELECT * FROM '../data/sales.csv')

Running query in 'duckdb:///:memory:'

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
product,VARCHAR,YES,None,None,None
category,VARCHAR,YES,None,None,None
quantity,BIGINT,YES,None,None,None
price,BIGINT,YES,None,None,None
order_date,DATE,YES,None,None,None
country,VARCHAR,YES,None,None,None
total,BIGINT,YES,None,None,None


**本数据结构(以 DESCRIBE 为准):**

order_id / customer_id / product / category / country → VARCHAR(条件要加单引号)

quantity / price / total → BIGINT(能做 SUM/AVG 等聚合)

order_date → DATE(真日期类型,可直接做日期运算 / STRFTIME)

- `CREATE OR REPLACE VIEW sales AS ...` 给查询起个短名字,后面 `FROM sales` 即可;
  视图**不存数据**,只存那段查询,永远和源 CSV 同步。

In [9]:
%%sql
CREATE OR REPLACE VIEW sales AS SELECT * FROM '../data/sales.csv';
SELECT COUNT(*) AS rows_count FROM sales

Running query in 'duckdb:///:memory:'

rows_count
500


## 1. 标量子查询

标量子查询 = 只返回一个值（1行1列）的子查询，可以当成一个普通的数字来用

In [10]:
%%sql
-- 全表平均金额
SELECT AVG(total) AS avg_total FROM sales;

Running query in 'duckdb:///:memory:'

avg_total
2535.432


In [11]:
%%sql
-- 找出金额高于“全表平均”的订单
-- WHERE里不能写AVG(total)，必须用子查询先把他查出来
SELECT order_id, total
FROM sales
WHERE total > (SELECT AVG(total) FROM sales)
ORDER BY total DESC;

Running query in 'duckdb:///:memory:'

order_id,total
O1110,9995
O1238,9995
O1470,9995
O1409,9995
O1152,9995
O1483,9995
O1432,9995
O1319,9995
O1324,9995
O1342,9995


关键点:Week 1 你知道「WHERE 里不能用聚合函数」。子查询就是解法——聚合在括号里算完,WHERE 拿到的是一个普通数字。

子查询也能放在 SELECT 里:

In [ ]:
%%sql
-- 每个订单顺便带上“全表平均”和差额
SELECT
    order_id,
    total,
    (SELECT AVG(total) FROM sales) AS avg_total,
    total - (SELECT AVG(total) FROM sales) AS diff
FROM sales
LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,total,avg_total,diff
O1000,2598,2535.432,62.56800000000021
O1001,99,2535.432,-2436.432
O1002,396,2535.432,-2139.432
O1003,396,2535.432,-2139.432
O1004,495,2535.432,-2040.4319999999998
O1005,198,2535.432,-2337.432
O1006,6495,2535.432,3959.568
O1007,1198,2535.432,-1337.4319999999998
O1008,1495,2535.432,-1040.4319999999998
O1009,297,2535.432,-2238.432


## 2. IN子查询
IN后面除了手写列表IN('A','B')，还能跟一个子查询返回的一列。

In [13]:
%%sql
-- 先单独看：哪些国家的客户数>=5?
SELECT country
FROM sales
GROUP BY country
HAVING COUNT(DISTINCT customer_id) >=5;

Running query in 'duckdb:///:memory:'

country
Germany
US
UK
France
China


In [14]:
%%sql
-- 把上面的结果当成“国家清单”，查这些国家的全部订单
SELECT order_id, country, total
FROM sales
WHERE country IN (
    SELECT country
    FROM sales
    GROUP BY country
    HAVING COUNT(DISTINCT customer_id) >= 5
);

Running query in 'duckdb:///:memory:'

order_id,country,total
O1000,Germany,2598
O1001,US,99
O1002,US,396
O1003,US,396
O1004,France,495
O1005,UK,198
O1006,UK,6495
O1007,UK,1198
O1008,UK,1495
O1009,Germany,297


💡 子查询的典型用法:先用一个查询圈出一批 key,再用主查询捞这批 key 的明细。

⚠️ 注意 HAVING 里写的是 COUNT(DISTINCT customer_id) 本身,不是别名——你弱点清单 #15。

## 3. EXISTS/相关子查询
前面的子查询都能独立运行(不相关子查询)。相关子查询的内层引用了外层的列,必须跟着外层逐行跑。

In [ ]:
%%sql
-- EXISTS: 外层每一行，去内层查“有没有满足条件的行”，有就保留
-- 找出“至少下过一笔total > 500 订单”的客户的所有订单
SELECT s1.order_id, s1.customer_id, s1.total
FROM sales s1
WHERE EXISTS(
    SELECT 1
    FROM sales s2
    WHERE s2.customer_id = s1.customer_id -- 引用了外层s1，所以“相关”
       AND s2.total > 500
);

Running query in 'duckdb:///:memory:'

order_id,customer_id,total
O1000,C007,2598
O1001,C004,99
O1002,C005,396
O1003,C007,396
O1004,C003,495
O1005,C008,198
O1006,C005,6495
O1007,C005,1198
O1008,C007,1495
O1009,C002,297


EXISTS 的心智模型:SELECT 1 里写什么不重要,EXISTS 只关心子查询有没有返回行——有→TRUE,无→FALSE。

相关子查询也常放 SELECT 里做「每行 vs 它所属分组」的对比:

In [20]:
%%sql
-- 每个订单的金额 vs 它所在国家的平均金额
SELECT
    s1.order_id,
    s1.couNtry,
    s1.total,
    (SELECT AVG(s2.total)
    FROM sales s2
    WHERE s2.country = s1.country) AS country_avg
FROM sales s1
LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,country,total,country_avg
O1000,Germany,2598,2182.121212121212
O1001,US,99,2404.904
O1002,US,396,2404.904
O1003,US,396,2404.904
O1004,France,495,2602.0625
O1005,UK,198,2714.807106598985
O1006,UK,6495,2714.807106598985
O1007,UK,1198,2714.807106598985
O1008,UK,1495,2714.807106598985
O1009,Germany,297,2182.121212121212


⚠️ 相关子查询「逐行执行」,数据量大时慢。Week 4 学窗口函数后这类需求有更优雅的写法,现在先理解逻辑。

## 4. FROM子查询（派生表）
子查询放在 FROM 后面,当成一张临时表用。这是「两步聚合」的标准解法。

In [21]:
%%sql
-- 需求：找出“消费总额”高于“全体客户平均消费总额”的客户
-- 第一步： 每个客户的消费总额（聚合结果， 当作一张临时表）
-- 第二步： 对这张临时表再聚合 + 过滤
SELECT customer_id, customer_total
FROM(
    SELECT customer_id, SUM(total) AS customer_total
    FROM sales
    GROUP BY customer_id
) AS t -- 派生表必须起别名
WHERE customer_total > (
    SELECT AVG(customer_total)
    FROM (
        SELECT customer_id, SUM(total) AS customer_total
        FROM sales
        GROUP BY customer_id
    ) AS t2
)
ORDER BY customer_total DESC;

Running query in 'duckdb:///:memory:'

customer_id,customer_total
C006,198806
C001,184001
C008,164484
C003,160212


感觉到重复了吗?同一段子查询写了两遍——这正是 Week 5 CTE(WITH ... AS)要解决的痛点。今天先忍受这个丑陋。

## 5. NOT IN的NULL陷阱⚠️（今日最重要）
高频面试坑,直接关联你 Day 4 学的「NULL 比较」。

In [ ]:
%%sql
-- 造刁钻测试用例（呼应弱点 #19）
-- 构造测试数据表，用于验证SQL空值NULL查询易错特性
-- 1.创建demo表，仅含id字段，存储基础数据1、2、3
-- 2.创建blocked黑名单表，包含数值2与空值NULL
-- t为临时数据集别名，仅查询内部生效，最终生成正式表名demo、blocked
CREATE OR REPLACE TABLE demo AS
    SELECT * FROM (VALUES (1), (2), (3)) AS t(id);

CREATE OR REPLACE TABLE blocked AS
    SELECT * FROM (VALUES (2), (NULL)) AS t(id); -- 注意这里有个NULL

Running query in 'duckdb:///:memory:'

Count


In [24]:
%%sql
-- 期望：返回id = 1 和 3
SELECT id FROM demo
WHERE id NOT IN (SELECT id FROM blocked);

Running query in 'duckdb:///:memory:'

id


结果:返回 0 行! 不是 SQL 写错了。

原因:id NOT IN (2, NULL) 展开成 id != 2 AND id != NULL。而 id != NULL 的结果是 UNKNOWN(不是 TRUE),整个 AND 永远无法为 TRUE,一行都过不了。

正确写法——用 NOT EXISTS:

In [25]:
%%sql
SELECT d.id
FROM demo d
WHERE NOT EXISTS (
    SELECT 1 FROM blocked b WHERE b.id = d.id
);

Running query in 'duckdb:///:memory:'

id
1
3


NOT EXISTS 不受 NULL 影响,正确返回 1 和 3。

铁律:子查询的列可能有 NULL 时,永远用 NOT EXISTS,别用 NOT IN。今晚进弱点清单。